# Provisioning resources

## Prerequisites
* You have successfully followed the [Preparation tutorial](./preparation.ipynb)
* You have basic Python knowledge

## Introduction

5G experiments are deployed on resources, they can be of various different
forms.

To set in place the resources, resource providers are provided. Their role is
to make sure that the resource is configured properly to receive 5G workloads.

In order to do so, provisioners take as input a metamodel, that describes the
5G network, and a deployement model that provides all technical information
regarding the actual deployement of the 5G network.

The particularity of the metamodel is that it is implementation agnostic. This
allows one to share their 5G information and experiment in various environments
and implementations.

Our orchestrator library is used to instantiate metamodels to these specific
environments. To do so, it complements the metamodel with a **deployment model**
that carries information about a specific deployment of the metamodel.

Provisioners are used to explicitly announce the nodes on which the functions
must be deployed. For that, we have to list these machines and tag them with
the roles that can support in the `nodes` section of the deployment model. The
library will automatically schedule the functions on the nodes with the roles
compatible with the functions.

The orchestrator offers a collection of provisionners that can be used to
provision machines and each such provisioner has its own set of configuration
parameters.

This tutotial is only about provisioning the infrastrcuture and we will not
deploy any 5G network. Nevertheless, provisioners always require a metamodel, so
we will instantiate an empty metamodel that we will use throughout this
tutorial.

In [1]:
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel()

The provisionners are manipulated via managers instantiated from the class
`ProvisioningManagement` that will automatically determine how to provision the
resources based on the deployement model.

## `null` Provisioner

The `null` provisioner is a specific case that indicates that no provisioner is
requested for the nodes declared with this provisioner. Instead, it is assumed
that the nodes are up and running and properly configured to accept workload
orchestrated by our library.

As there is not provider operations, we have to manually specify that the node
is usable by the library, this is done by setting the `is_provisioned` attribute
to true. When using other provisioners, users don't have to worry about this
attribute as it is the provisioner itself that populates it according to the
status of the provisioning.

The library always uses SSH to access the node. In the example below the
`default.example.org` node is accessed using default SSH configuration. By
default the username is `root` and the SSH client uses the defaults of the local
SSH client of the users running the library.

The `password.example.org` node shows the case where the machine is accessed via
user `ubuntu` with the default SSH keys. As the node requires a sudo
password, the configuration specifies the sudo password for the user `ubuntu`.

The `key.example.org` node is not accessible by using the default SSH keys of
the local user, instead a specific private is provided.

**Important:** when the `null` provisioner is used, the name of the node must
in the metamodel be the actual node's hostname. The library will use the name
as host to connect to with SSH.

In [2]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
nodes:
  default.example.org: # node's actual hostname
    is_provisioned: true
    provisioner: null

  password.example.org: # node's actual hostname
    is_provisioned: true
    provisioner: null
    ssh_info:
      user: ubuntu
      sudo_password: password

  key.example.org: # node's actual hostname
    is_provisioned: true
    ssh_info:
      private_key_file: /home/user/.ssh/id_ed25519_custom
"""
)

As the `null` provisioner does not really exist, there is nothing else to do as
long as the node is declared `is_provisioned`.

**NOTE:** it is of the responsibility of the users to ensure that nodes that
they provisioned by themselves are compliant with the library.

## SLICES BI Provisioner

The SLICES BI provisioner is used to create ephemeral virtual machines on the
[SLICES Basic Infrastructure Service (SLICES BI)](https://doc.slices-ri.eu/BasicServices/BasicInfrastructure/basicinfrastructure.html).

The SLICES BI provisioner requires that the SLICES CLI is installed on your
machine. Refer to the [Preparation tutorial](./preparation.ipynb) for details on
how to install it and learn the fundamental SLICES-RI concepts.

**IMPORTANT NOTE:** make sure your SSH public keys are registered to SLICES
before deploying your network.

We already defined a metamodel, now we have to define the deployment model
containing information about the infrastructure we want to provision.

The manager of type `ProvisioningManagement` role is to provision the machines
found in the `nodes` section of the deployment model. It automatically
determines which exact provisionner to use for each node. The information about
the provisioners is defined in the `provisioners` section of the deployment
model.

In this example we will deploy 3 virtual machines (`node1`, `node2`, and
`node3`) in the `fr-sophia2-bi-vm1` SLICES-BI infrastructure. These machines
will be deployed in the experiment `provision_tutorial` within the `post5g-beta`
project. We ask the provisioner to automatically create the experiment for us
and nodes will have a lifetime of 8h after which they will be automatically
destroyed. We will then define a provisionner, that we will call
`my-provisioner` with these parameters, in the `provisioners` section of the
deployment model.


Once the provisioner is defined, we can specify the nodes and link them to the
provisioner to use to deploy them. In this example, `node1` will be tagged with
the role `core`, `node2` with the role `ran`, and `node3` will accept both
`ran` and `ue` roles. They are all provisioned with the `my-provisioner`
provisioner.


In [ ]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
provisioners:
  my-provisionner:
    type: bi
    params:
      infra: fr-sophia2-bi-vm1
      project: post5g-beta
      experiment: provision_tutorial
      create_experiment: true
      experiment_duration: 8h
      node_duration: 8h
nodes:
  node1:
    provisioner: my-provisionner
    roles:
    - core
  node2:
    provisioner: my-provisionner
    roles:
    - ran
  node3:
    provisioner: my-provisionner
    roles:
    - ran
    - ue
"""
)


To provision the nodes we just have to create an instance of a
`ProvisioningManagement` manager with the metamodel and deployment model we
defined above and then to call its `provision` function.

In [4]:
from orchestrator.deployment import ProvisioningManagement
prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision(quiet=False)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node1' provisioning] ***

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for '

We now have our three virtual machines deployed in the SLICES BI and configured
according to the needs of this scenario. We can verify it by listing the
resources of the experiment with the SLICES CLI (adapt the parameters to your
setup).

In [5]:
!SLICES_BI_SITE_ID=fr-sophia2-bi-vm1 slices bi list-resources --experiment provision_tutorial

  Resources in experiment exp_expauth.ilabt.imec.be_01m4krjd21f1fr8ddw2enmhf05  
┏━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃           ┃       ┃        ┃ Created   ┃            ┃ Public    ┃ Private    ┃
┃ ID        ┃ Name  ┃ Status ┃ At        ┃ Expires At ┃ IPv4      ┃ IPv4       ┃
┡━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ r_fr-sop… │ node3 │ up     │ 2026-10-… │ 2026-10-10 │           │ 10.123.10… │
│           │       │        │ 20:37 UTC │ 21:37 UTC  │           │            │
│ r_fr-sop… │ node1 │ up     │ 2026-10-… │ 2026-10-10 │           │ 10.123.23… │
│           │       │        │ 20:37 UTC │ 21:37 UTC  │           │            │
│ r_fr-sop… │ node2 │ up     │ 2026-10-… │ 2026-10-10 │           │ 10.123.59… │
│           │       │        │ 20:37 UTC │ 21:37 UTC  │           │            │
└───────────┴───────┴────────┴───────────┴────────────┴───────────┴────────────┘


## Save metamodel and deployment model

Each manager consumes a metamodel and a deployment model and  the result of its
actions is the creation of a new deployment model that can be used by the manager
next in line.

Many different methods exist to obtain the metamodel and/or  deployment model
of a manager. For example the `output()` method directly returns the two models.
As you expected, the deployment model contains all the information to acces the
freshly created machines.

We can save the metamodel and deployment model corresponding to our action.

In [7]:
metamodel, deployment_model = prov.output()
from pathlib import Path
DIR="state/provision"
Path(DIR).mkdir(parents=True, exist_ok=True)

metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml", exclude_defaults=False)
deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml", exclude_defaults=False)

## Unprovision nodes

The provisioners can also be used to unprovision the nodes they have
provisioned.

You can either provide the list of nodes to be unprovisioned or unprovision all
nodes of the deployment model.

The effect of unprovisioning a node depends on the provisioner (e.g., the
`SLICES BI` provisioner destroys the VM corresponding to the node) but in any
case the node remains present in the deployement model (and so could be
re-provisioned later) but its attribute `is_provisioned` is set to false.

In [ ]:
await prov.unprovision_nodes(nodes=["node1", "node2"])  # Unprovision some nodes
await prov.unprovision()                                # Unprovision all nodes